In [ ]:
import numpy as np 
import matplotlib.pyplot as plt
import pandas as pd

from scipy.optimize import curve_fit

# A longer Time


In [ ]:
MC_kappa = pd.read_csv("kappa_new.csv", header=None, index_col=None)
MC_regret = pd.read_csv("regret_new.csv", header=None, index_col=None)

MC_kappa = np.array(MC_kappa)
MC_regret = np.array(MC_regret)

In [ ]:
kappa_true = 10
Regret_MC = 200
T=1000
dt=1

ts = np.linspace(0, T, int(T/dt)+1)

std = np.std(MC_kappa-kappa_true, axis=0)
ci = 1.64 * std/np.sqrt(Regret_MC)
y = np.mean(MC_kappa, axis=0)-kappa_true

fig, ax = plt.subplots()
ax.loglog(ts, y)
ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1)

# plt.loglog(ts[1:], ts[1:]**(-0.5), '--')
plt.xlabel("Log Time")
plt.ylabel("Log Error")
plt.title("Log of Learning Error")
plt.show()

In [ ]:
std = np.std(MC_regret, axis=0)
ci = 1.96 * std/np.sqrt(Regret_MC)
y = np.mean(MC_regret, axis=0)

fig, ax = plt.subplots()
ax.plot(ts, y)
ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1)
plt.xlabel("Time")
plt.ylabel("Regret")
plt.title("Algorithm Regret")
plt.show()

In [ ]:
start = 2

y_axis = (y)[start:]
y_axis_upper = (y+ci)[start:]
x_axis = ts[start:]

# Define the quadratic model
def log_square_model(x, c, m):
    return c * (np.log(x)**2) + m

def calculate_r2(data, fit_data):
    residual = data - fit_data
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((data - np.mean(data))**2)
    r_2 = 1 - ss_res / ss_tot
    return r_2

def log_model(x,c, m):
    return c * np.log(x) + m

# Use log^2 model
params, covariance = curve_fit(log_square_model, x_axis, y_axis)
c_estimated = params[0]
m_estimated = params[1]

params_upper, covariance_upper = curve_fit(log_square_model, x_axis, y_axis_upper)
c_estimated_upper = params_upper[0]
m_estimated_upper = params_upper[1]



# Use log model 
params_log, covariance_log = curve_fit(log_model, x_axis, y_axis)
c_estimated_log = params_log[0]
m_estimated_log = params_log[1]


params_upper_log, covariance_upper_log = curve_fit(log_model, x_axis, y_axis_upper)
c_estimated_upper_log = params_upper_log[0]
m_estimated_upper_log = params_upper_log[1]

In [ ]:
fig, ax = plt.subplots()
ax.plot(ts, y, label='expected regret')
ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1, label='$95\%$ CI of regret')

x_fit = ts[start:]
y_fit = log_square_model(x_fit, c_estimated, m_estimated)
y_fit_upper = log_square_model(x_fit, c_estimated_upper, m_estimated_upper)

y_fit_log = log_model(x_fit, c_estimated_log, m_estimated_log)
y_fit_upper_log = log_model(x_fit, c_estimated_upper_log, m_estimated_upper_log)

ax.plot(x_fit, y_fit, color='mediumslateblue',linestyle=':', label=f'Curve ${np.round(c_estimated, 3)} \ln^2 T + {np.round(m_estimated,4)}, R^2 = {np.round(calculate_r2(y_axis, y_fit),3)}$')
ax.plot(x_fit, y_fit_log, color='violet',linestyle=':', label=f'Curve ${np.round(c_estimated_log, 3)} \ln T   {np.round(m_estimated_log,4)}, R^2 = {np.round(calculate_r2(y_axis, y_fit_log),3)}$')


ax.plot(x_fit, y_fit_upper, color='mediumslateblue',linestyle='dashdot', label=f'Curve ${np.round(c_estimated_upper, 4)} \ln^2 T + {np.round(m_estimated_upper,4)},  R^2 = {np.round(calculate_r2(y_axis_upper, y_fit_upper),3)}$')
ax.plot(x_fit, y_fit_upper_log, color='violet',linestyle='dashdot', label=f'Curve ${np.round(c_estimated_upper_log, 4)} \ln T {np.round(m_estimated_upper_log,4)}, R^2 = {np.round(calculate_r2(y_axis_upper, y_fit_upper_log),3)}$')

plt.xlabel("Time")
plt.ylabel("Regret")
plt.title("Algorithm Regret")
plt.legend()
plt.show()

# Benchmark

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

MC_kappa = pd.read_csv("kappa_regularised.csv", header=None, index_col=None)
MC_regret = pd.read_csv("regret_regularised.csv", header=None, index_col=None)

MC_kappa_benchmark = pd.read_csv("kappa_benchmark2.csv", header=None, index_col=None)
MC_regret_benchmark = pd.read_csv("regret_benchmark2.csv", header=None, index_col=None)

In [ ]:
dt=1
Regret_MC = 200
time_steps = MC_kappa.shape[1]
ts = np.linspace(0, (time_steps-1)/dt, time_steps)
kappa_true = 10


In [ ]:
std = np.std(MC_kappa-kappa_true, axis=0)
std_benchmark = np.std(MC_kappa_benchmark-kappa_true, axis=0)
ci = 1.64 * std/np.sqrt(Regret_MC)
ci_benchmark = 1.64 * std_benchmark/np.sqrt(Regret_MC)
y = np.mean(MC_kappa, axis=0)-kappa_true
y_benchmark = np.mean(MC_kappa_benchmark, axis=0)-kappa_true

fig, ax = plt.subplots()
ax.loglog(ts, y, label='optimal strategy')
ax.loglog(ts, y_benchmark, label='myopic strategy')
# ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1)

# plt.loglog(ts[1:], ts[1:]**(-0.5), '--')
plt.xlabel("Log Time")
plt.ylabel("Log Error")
plt.title("Log of Learning Error")
plt.legend()
plt.show()

# plt.savefig("/Users/galen/Desktop/learning_error_benchmark.pdf", format='pdf', bbox_inches='tight')
# plt.close()


In [ ]:
std = np.std(MC_regret, axis=0)
std_benchmark = np.std(MC_regret_benchmark, axis=0)
ci = 1.96 * std/np.sqrt(Regret_MC)
ci_benchmark = 1.96 * std_benchmark/np.sqrt(Regret_MC)
y = np.mean(MC_regret, axis=0)
y_benchmark = np.mean(MC_regret_benchmark, axis=0)

fig, ax = plt.subplots()
ax.plot(ts, y, label='optimal strategy')
ax.plot(ts, y_benchmark, label='myopic strategy')
# ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1)
plt.xlabel("Time")
plt.ylabel("Regret")
plt.title("Algorithm Regret")
plt.legend()
plt.show()

# plt.savefig("/Users/galen/Desktop/regret_benchmark.pdf", format='pdf', bbox_inches='tight')
# plt.close()

In [ ]:
start = 10

y_axis = (y)[start:]
y_axis_benchmark = (y_benchmark)[start:]
y_axis_upper = (y+ci)[start:]
y_axis_benchmark_upper = (y_benchmark+ci_benchmark)[start:]
x_axis = ts[start:]
y_axis_benchmark = (y_benchmark)[start:]

# Define the quadratic model
def log_square_model(x, c, m):
    return c * (np.log(x)**2) + m

def calculate_r2(data, fit_data):
    residual = data - fit_data
    ss_res = np.sum(residual**2)
    ss_tot = np.sum((data - np.mean(data))**2)
    r_2 = 1 - ss_res / ss_tot
    return r_2

def log_model(x,c, m):
    return c * np.log(x) + m

# Use log^2 model
params, covariance = curve_fit(log_square_model, x_axis, y_axis)
c_estimated = params[0]
m_estimated = params[1]

params_upper, covariance_upper = curve_fit(log_square_model, x_axis, y_axis_upper)
c_estimated_upper = params_upper[0]
m_estimated_upper = params_upper[1]



# Use log model 
params_log, covariance_log = curve_fit(log_model, x_axis, y_axis)
c_estimated_log = params_log[0]
m_estimated_log = params_log[1]


params_upper_log, covariance_upper_log = curve_fit(log_model, x_axis, y_axis_upper)
c_estimated_upper_log = params_upper_log[0]
m_estimated_upper_log = params_upper_log[1]

fig, ax = plt.subplots()
ax.plot(ts, y, label='expected regret')
ax.fill_between(ts, (y-ci), (y+ci), color='b', alpha=.1, label='$95\%$ CI of regret')

x_fit = ts[start:]
y_fit = log_square_model(x_fit, c_estimated, m_estimated)
y_fit_upper = log_square_model(x_fit, c_estimated_upper, m_estimated_upper)

y_fit_log = log_model(x_fit, c_estimated_log, m_estimated_log)
y_fit_upper_log = log_model(x_fit, c_estimated_upper_log, m_estimated_upper_log)

ax.plot(x_fit, y_fit, color='mediumslateblue',linestyle=':', label=f'Curve ${np.round(c_estimated, 3)} \ln^2 T + {np.round(m_estimated,4)}, R^2 = {np.round(calculate_r2(y_axis, y_fit),3)}$')
ax.plot(x_fit, y_fit_log, color='violet',linestyle=':', label=f'Curve ${np.round(c_estimated_log, 3)} \ln T   {np.round(m_estimated_log,4)}, R^2 = {np.round(calculate_r2(y_axis, y_fit_log),3)}$')


ax.plot(x_fit, y_fit_upper, color='mediumslateblue',linestyle='dashdot', label=f'Curve ${np.round(c_estimated_upper, 4)} \ln^2 T + {np.round(m_estimated_upper,4)},  R^2 = {np.round(calculate_r2(y_axis_upper, y_fit_upper),3)}$')
ax.plot(x_fit, y_fit_upper_log, color='violet',linestyle='dashdot', label=f'Curve ${np.round(c_estimated_upper_log, 4)} \ln T {np.round(m_estimated_upper_log,4)}, R^2 = {np.round(calculate_r2(y_axis_upper, y_fit_upper_log),3)}$')

plt.xlabel("Time")
plt.ylabel("Regret")
plt.title("Algorithm Regret")
plt.legend()

plt.show()

In [ ]:
# Compute statistics
std = np.std(MC_kappa - kappa_true, axis=0)
std_benchmark = np.std(MC_kappa_benchmark - kappa_true, axis=0)
ci = 1.64 * std / np.sqrt(Regret_MC)
ci_benchmark = 1.64 * std_benchmark / np.sqrt(Regret_MC)
y = np.mean(MC_kappa, axis=0) - kappa_true
y_benchmark = np.mean(MC_kappa_benchmark, axis=0) - kappa_true

std_regret = np.std(MC_regret, axis=0)
std_regret_benchmark = np.std(MC_regret_benchmark, axis=0)
ci_regret = 1.96 * std_regret / np.sqrt(Regret_MC)
ci_regret_benchmark = 1.96 * std_regret_benchmark / np.sqrt(Regret_MC)
y_regret = np.mean(MC_regret, axis=0)
y_regret_benchmark = np.mean(MC_regret_benchmark, axis=0)

# Create subplots
fig, axs = plt.subplots(1, 2, figsize=(12, 5))

# Plot learning error
axs[0].loglog(ts, y, label='log regret algortihm')
axs[0].loglog(ts, y_benchmark, label='myopic algortihm')
axs[0].set_xlabel("Log Time (s)")
axs[0].set_ylabel("Log Error")
axs[0].set_title("Log of learning error of $\kappa$")
axs[0].legend()

# Plot regret
axs[1].plot(ts, y_regret, label='log regret algortihm')
axs[1].plot(ts, y_regret_benchmark, label='myopic algortihm')
axs[1].set_xlabel("Time (s)")
axs[1].set_ylabel("Regret (\$)")
axs[1].set_title("Algorithm regret")
axs[1].legend()

plt.tight_layout()

plt.savefig("/Users/galen/Desktop/benchmark.pdf", format='pdf', bbox_inches='tight')
plt.show()
plt.close()


## Compare the running time

In [ ]:
from lib.ECP import Agent
import time

kappa_true = 10
kappa_est = np.random.uniform(1, 100)

lambda_buy = 0.4
lambda_sell = 0.4
q_upper, q_lower = 30, -30
phi = 1e-6

simga = 0.01

T=100
dt=1
K_upper=100
K_lower=0.1
ts = np.linspace(0, T, int(T/dt)+1)
kappa_est = np.random.uniform(50, 100)

In [ ]:
# run time
test_samples = 50
start_time = time.time()

for _ in range(test_samples):

    agent = Agent(
        lambda_buy=lambda_buy,
        lambda_sell=lambda_sell,
        q_upper=q_upper,
        q_lower=q_lower,
        phi=phi,
        kappa_est=kappa_est,
        T=T,
        dt=dt,
        K_upper=K_upper,
        K_lower=K_lower, 
    )

    agent.learning(sigma=simga, kappa_true=kappa_true)

end_time = time.time()
print(f"Execution time for the full algorithm: {(end_time - start_time)/test_samples:.2f} seconds")

In [ ]:
start_time = time.time()

for _ in range(test_samples):

    agent_benchmark = Agent(
        lambda_buy=lambda_buy,
        lambda_sell=lambda_sell,
        q_upper=q_upper,
        q_lower=q_lower,
        phi=phi,
        kappa_est=kappa_est,
        T=T,
        dt=dt,
        K_upper=K_upper,
        K_lower=K_lower, 
    )

    agent_benchmark.learning_myopic(sigma=simga, kappa_true=kappa_true)

end_time = time.time()
print(f"Execution time for the benchmark: {(end_time - start_time)/test_samples:.2f} seconds")

# Non-stationary market

In [ ]:
from lib.ECP import Agent_nonstationary, ErgodicCP
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
kappa_true_list = [20, 30, 10, 40, 25]
# kappa_true_list = [10, 40, 20]
non_stationary_interval = 50

T = len(kappa_true_list) * non_stationary_interval
dt = 1
ts = np.linspace(0, T, int(T/dt)+1)

kappa_est = np.random.uniform(1, 60)

lambda_buy = 0.3
lambda_sell = 0.3
q_upper, q_lower = 30, -30
phi = 1e-3
K_upper=100
K_lower=0.1

simga = 0.01

Regret_MC = 2000

In [ ]:
gamma_list = []
for kappa_true in kappa_true_list:
    gamma = ErgodicCP(
        lambda_buy=lambda_buy,
        lambda_sell=lambda_sell,
        q_upper=q_upper,
        q_lower=q_lower,
        phi=phi,
        kappa=kappa_true,
    ).EConst
    gamma_list.append(gamma)

optimal_reward = np.zeros(len(ts))
for idx, gamma in enumerate(gamma_list):
    # Calculate the start and end indices for the current interval
    start_idx = int(idx * (non_stationary_interval / dt))
    end_idx = min(int((idx + 1) * (non_stationary_interval / dt)), len(ts))  # Ensure end_idx doesn't exceed len(ts)

    # Assign the optimal reward for the current interval
    if idx == 0:
        # For the first interval, start from 0
        optimal_reward[start_idx:end_idx] = gamma * ts[start_idx:end_idx]
    else:
        # For subsequent intervals, continue from the previous reward
        optimal_reward[start_idx:end_idx] = (
            optimal_reward[start_idx - 1] + gamma * (ts[start_idx:end_idx] - ts[start_idx - 1])
        )

optimal_reward = optimal_reward + gamma_list[0]
optimal_reward[-1] = optimal_reward[-2] + gamma_list[-1]*(ts[-1]-ts[-2])

In [ ]:
MC_regret_EWMA = []
MC_kappa_EWMA  = []

for _ in range(Regret_MC):
    kappa_est = np.random.uniform(50, 100)

    agent = Agent_nonstationary(
        lambda_buy=lambda_buy,
        lambda_sell=lambda_sell,
        q_upper=q_upper,
        q_lower=q_lower,
        phi=phi,
        kappa_est=kappa_est,
        T=T,
        dt=dt,
        K_upper=K_upper,
        K_lower=K_lower, 
        non_stationary_interval=non_stationary_interval,
        method='EWMA',
        alpha=0.1,
    )

    agent.learning(sigma=simga, kappa_true_list=kappa_true_list)
    MC_kappa_EWMA.append(agent.kappa_learnlist)
    MC_regret_EWMA.append(optimal_reward - agent.objective)

MC_kappa_EWMA = np.array(MC_kappa_EWMA)
MC_regret_EWMA = np.array(MC_regret_EWMA)

In [ ]:
# df_kpp = pd.DataFrame(MC_kappa_EWMA)
# df_regret = pd.DataFrame(MC_regret_EWMA)

# df_kpp.to_csv("kappa_nonstationary_EWMA.csv", index=False, header=None)
# df_regret.to_csv("regret_nonstationary_EWMA.csv", index=False, header=None)

In [ ]:
MC_regret_SW = []
MC_kappa_SW = []

for _ in range(Regret_MC):
    kappa_est = np.random.uniform(50, 100)

    agent = Agent_nonstationary(
        lambda_buy=lambda_buy,
        lambda_sell=lambda_sell,
        q_upper=q_upper,
        q_lower=q_lower,
        phi=phi,
        kappa_est=kappa_est,
        T=T,
        dt=dt,
        K_upper=K_upper,
        K_lower=K_lower, 
        non_stationary_interval=non_stationary_interval,
        method='SW',
        window_size=30,
    )

    agent.learning(sigma=simga, kappa_true_list=kappa_true_list)
    MC_kappa_SW.append(agent.kappa_learnlist)
    MC_regret_SW.append(optimal_reward - agent.objective)

MC_kappa_SW = np.array(MC_kappa_SW)
MC_regret_SW = np.array(MC_regret_SW)

In [ ]:
# df_kpp_SW = pd.DataFrame(MC_kappa_SW)
# df_regret_SW = pd.DataFrame(MC_regret_SW)

# df_kpp_SW.to_csv("kappa_nonstationary_SW.csv", index=False, header=None)
# df_regret_SW.to_csv("regret_nonstationary_SW.csv", index=False, header=None)

In [ ]:
MC_regret_EWMA = pd.read_csv("regret_nonstationary_EWMA.csv", header=None, index_col=None)
MC_kappa_EWMA = pd.read_csv("kappa_nonstationary_EWMA.csv", header=None, index_col=None)
MC_regret_SW = pd.read_csv("regret_nonstationary_SW.csv", header=None, index_col=None)
MC_kappa_SW = pd.read_csv("kappa_nonstationary_SW.csv", header=None, index_col=None)

In [ ]:
std_ma = np.std(MC_regret_EWMA, axis=0)
std_sw = np.std(MC_regret_SW, axis=0)
# std_benchmark = np.std(MC_regret_benchmark, axis=0)
ci_ma = 1.96 * std_ma/np.sqrt(Regret_MC)
ci_sw = 1.96 * std_sw/np.sqrt(Regret_MC)
# ci_benchmark = 1.96 * std_benchmark/np.sqrt(Regret_MC)
y_ma = np.mean(MC_regret_EWMA, axis=0)
y_sw = np.mean(MC_regret_SW, axis=0)
# y_benchmark = np.mean(MC_regret_benchmark, axis=0)

fig, ax = plt.subplots()
ax.plot(ts, y_ma, label='exponentially weighted moving average')
ax.plot(ts, y_sw, label='sliding window')
# ax.plot(ts, y_benchmark)
# ax.fill_between(ts, (y_ma-ci_ma), (y_ma+ci_ma), color='b', alpha=.1)
plt.xlabel("Time")
plt.ylabel("Regret")
plt.title("Algorithm Regret of Non-stationary Market")
plt.legend()
plt.show()

# plt.savefig("/Users/galen/Desktop/regret_nonstaionary.pdf", format='pdf', bbox_inches='tight')
# plt.close()

In [ ]:
kappa_true_time = []
for kappa in kappa_true_list:
    kappa_true_time.extend([kappa]*non_stationary_interval)
kappa_true_time.append(kappa_true_list[-1])

plt.plot(ts, kappa_true_time, '--', label='true $\kappa$')
plt.plot(ts, np.mean(MC_kappa_EWMA, axis=0), label='estimated $\kappa (EWMA)$')
plt.plot(ts, np.mean(MC_kappa_SW, axis=0), label='estimated $\kappa (SW)$')


plt.xlabel("Time")
plt.ylabel("$\kappa$")
plt.title("Learn $\kappa$ in Non-stationary Market")
plt.legend()
plt.show()

# plt.savefig("/Users/galen/Desktop/kappa_learn_nonstaionary.pdf", format='pdf', bbox_inches='tight')
# plt.close()

In [ ]:
# --- Compute statistics ---
std_ma = np.std(MC_regret_EWMA, axis=0)
std_sw = np.std(MC_regret_SW, axis=0)
ci_ma = 1.96 * std_ma / np.sqrt(Regret_MC)
ci_sw = 1.96 * std_sw / np.sqrt(Regret_MC)
y_ma = np.mean(MC_regret_EWMA, axis=0)
y_sw = np.mean(MC_regret_SW, axis=0)

# --- Create subplots ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- First subplot: Regret over time ---
axes[1].plot(ts, y_ma, label='Exponentially Weighted Moving Average')
axes[1].plot(ts, y_sw, label='Sliding Window')
# Optionally include confidence intervals:
# axes[0].fill_between(ts, y_ma - ci_ma, y_ma + ci_ma, color='blue', alpha=0.1)
# axes[0].fill_between(ts, y_sw - ci_sw, y_sw + ci_sw, color='orange', alpha=0.1)
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Regret (\$)")
axes[1].set_title("Algorithm regret in non-stationary market")
axes[1].legend()

# --- Second subplot: kappa estimation over time ---
kappa_true_time = []
for kappa in kappa_true_list:
    kappa_true_time.extend([kappa]*non_stationary_interval)
kappa_true_time.append(kappa_true_list[-1])  # Extend to match ts length

axes[0].plot(ts, kappa_true_time, '--', label='True $\kappa$')
axes[0].plot(ts, np.mean(MC_kappa_EWMA, axis=0), label='Estimated $\kappa$ (EWMA)')
axes[0].plot(ts, np.mean(MC_kappa_SW, axis=0), label='Estimated $\kappa$ (SW)')

axes[0].set_xlabel("Time (s)")
axes[0].set_ylabel("$\kappa$")
axes[0].set_title("Learning $\kappa$ in non-stationary market")
axes[0].legend()

# --- Layout ---
plt.tight_layout()
# plt.show()

plt.savefig("/Users/galen/Desktop/nonstationary_sim.pdf", format='pdf', bbox_inches='tight')
plt.show()
plt.close()

# Dependency of regret constant $C$ on parameters

### $C_1$ with $\phi$ and $\lambda$

In [ ]:
from lib.ECP import Agent
import numpy as np
import seaborn as sns
from scipy.optimize import curve_fit
import pandas as pd
import matplotlib.pyplot as plt

def log_square_model(x, c, m):
    return c * (np.log(x)**2) + m

kappa_true = 10
kappa_est = np.random.uniform(1, 100)
q_upper, q_lower = 30, -30

simga = 0.01

T=100
dt=1
K_upper=100
K_lower=0.1
ts = np.linspace(0, T, int(T/dt)+1)
simga = 0.01
Regret_MC = 500
start = 10
seed = 42

# heat map 
lambda_list = np.linspace(0.1, 3, 8)
phi_list = np.linspace(1e-6, 1e-2, 8)

const = np.zeros((len(lambda_list), len(phi_list)))

In [ ]:
for idxlambda, lambda_ in enumerate(lambda_list):
    for idxphi, phi in enumerate(phi_list): 
        regret_list = []

        np.random.seed(seed)
        for _ in range(Regret_MC):
            agent = Agent(
                lambda_buy=lambda_,
                lambda_sell=lambda_,
                q_upper=q_upper,
                q_lower=q_lower,
                phi=phi,
                kappa_est=kappa_est,
                T=T,
                dt=dt,
                K_upper=K_upper,
                K_lower=K_lower, 
            )

            agent.learning(sigma=simga, kappa_true=kappa_true)
            regret_list.append(agent.regret(kappa_true=kappa_true))

        regret_list = np.array(regret_list)
        y = np.mean(regret_list, axis=0)
        y_axis = (y)[start:]
        x_axis = ts[start:]

        # Use log^2 model fit curve
        params, covariance = curve_fit(log_square_model, x_axis, y_axis)
        const[idxlambda, idxphi] = params[0]

df = pd.DataFrame(const, index=lambda_list, columns=phi_list)
df.to_csv("C1_phi_lambda_new.csv")

In [ ]:
df_read = pd.read_csv("C1_phi_lambda_new.csv", index_col=0)
const_phi_lambda = np.array(df_read)

In [ ]:
sns.heatmap(df_read, cmap='YlGnBu', annot=False, vmin=np.min(const_phi_lambda), vmax=np.max(const_phi_lambda))
# Customize ticks to show numbers with 3 decimal places

plt.yticks(
    ticks=np.arange(0, len(lambda_list), max(1, len(lambda_list) // 10)),  # Adjust tick frequency
    labels=[f"{lam:.3f}" for lam in lambda_list[::max(1, len(lambda_list) // 10)]]
)

plt.xticks(
    ticks=np.arange(0, len(phi_list), max(1, len(phi_list) // 10)),  # Adjust tick frequency
    labels=[f"{phi:.3f}" for phi in phi_list[::max(1, len(phi_list) // 10)]],
    rotation=45
)



# Set axis labels
plt.ylabel("$\lambda$")  # Label the x-axis as "k"
plt.xlabel("$\phi$")  # Label the y-axis as "r"
plt.title("Dependence of fitted $C_1$ on model parameters")
plt.show()

plt.savefig("/Users/galen/Desktop/C1_phi_lambda.pdf", format='pdf', bbox_inches='tight')
# plt.close()

### $C_1$ with $\bar K$ and $\kappa_0$

In [ ]:
kappa_true = 10
q_upper, q_lower = 30, -30

simga = 0.01

T=100
dt=1
K_upper=100
K_lower= 1/ K_upper
ts = np.linspace(0, T, int(T/dt)+1)
simga = 0.01

lambda_buy, lambda_sell = 0.4, 0.4
phi = 1e-4
Regret_MC = 500
start = 10
# heat map 
kappa_est_list = [20, 40, 60, 80, 100]
K_upper_list= [20, 40, 60, 80, 100]


const = np.zeros((len(kappa_est_list), len(K_upper_list)))
# const_2 = np.zeros((len(kappa_est_list), len(K_upper_list)))

In [ ]:
for idxkest, kappa_est in enumerate(kappa_est_list):
    for idxKbar, K_upper in enumerate(K_upper_list):
        regret_list = []
        K_lower = 1 / K_upper
        np.random.seed(seed)
        for _ in range(Regret_MC):
            agent = Agent(
                lambda_buy=lambda_buy,
                lambda_sell=lambda_sell,
                q_upper=q_upper,
                q_lower=q_lower,
                phi=phi,
                kappa_est=kappa_est,
                T=T,
                dt=dt,
                K_upper=K_upper,
                K_lower=K_lower, 
            )

            agent.learning(sigma=simga, kappa_true=kappa_true)
            regret_list.append(agent.regret(kappa_true=kappa_true))

        regret_list = np.array(regret_list)
        y = np.mean(regret_list, axis=0)
        y_axis = (y)[start:]
        x_axis = ts[start:]

        # Use log^2 model fit curve
        params, covariance = curve_fit(log_square_model, x_axis, y_axis)
        const[idxkest, idxKbar] = params[0]
        # const_2[idxkest, idxKbar] = params[1]


In [ ]:
df_2 = pd.DataFrame(const, index=kappa_est_list, columns=K_upper_list)
df_2.index = np.array(kappa_est_list) - kappa_true 
df_2.to_csv("C1_kappaest_barK_new.csv")

In [ ]:
# df_2 = pd.read_csv("C1_kappaest_barK.csv")
# df.index = np.array(kappa_est_list) - kappa_true
# const = np.array(df)

In [ ]:
mask = np.triu(np.ones_like(df_2, dtype=bool), k=1).T  # Mask for strictly upper triangular part

sns.heatmap(df_2, cmap='YlGnBu', mask=mask, annot=False, vmin=np.min(const), vmax=np.max(const))
# sns.heatmap(df_2, cmap='YlGnBu', annot=False, vmin=np.min(const), vmax=np.max(const))

# Set axis labels
plt.xlabel("$\\bar{K}$")  # Label the x-axis as "k"
plt.ylabel("$\kappa_0 - \kappa^{\\ast}$, with $\kappa^{\\ast} = 10$")  # Label the y-axis as "r"
plt.title("Dependence of fitted $C_1$ on model parameters")
plt.show()

plt.savefig("/Users/galen/Desktop/C1_K_kappaest.pdf", format='pdf', bbox_inches='tight')
# plt.close()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # 1 row, 2 columns

# --- First heatmap ---
sns.heatmap(df_read, cmap='YlGnBu', annot=False, vmin=np.min(const_phi_lambda), vmax=np.max(const_phi_lambda), ax=axes[0])

axes[0].set_yticks(
    ticks=np.arange(0, len(lambda_list), max(1, len(lambda_list) // 10))
)
axes[0].set_yticklabels(
    [f"{lam:.3f}" for lam in lambda_list[::max(1, len(lambda_list) // 10)]]
)

axes[0].set_xticks(
    ticks=np.arange(0, len(phi_list), max(1, len(phi_list) // 10))
)
axes[0].set_xticklabels(
    [f"{phi:.3f}" for phi in phi_list[::max(1, len(phi_list) // 10)]],
    rotation=45
)

axes[0].set_ylabel("$\lambda$")
axes[0].set_xlabel("$\phi$")
axes[0].set_title("Dependence of fitted $C_1$ on model parameters")

# --- Second heatmap ---
mask = np.triu(np.ones_like(df_2, dtype=bool), k=1).T
sns.heatmap(df_2, cmap='YlGnBu', annot=False, vmin=np.min(const), vmax=np.max(const), mask=mask, ax=axes[1])

axes[1].set_xlabel("$\\bar{K}$")
axes[1].set_ylabel("$\kappa_0 - \kappa^{\\ast}$, with $\kappa^{\\ast} = 10$")
axes[1].set_title("Dependence of fitted $C_1$ on model parameters")

plt.tight_layout()
# plt.show()

plt.savefig("/Users/galen/Desktop/C1_model_parameters.pdf", format='pdf', bbox_inches='tight')
plt.show()
plt.close()


### $C_1$ with $\bar q$ and $\underline q$

In [ ]:
kappa_true = 10
# heat map
q_upper_list, q_lower_list = [5, 10, 15, 20, 25, 30, 35], [-5, -10, -15, -20, -25, -30, -35]

simga = 0.01

T=75
dt=1
K_upper=100
K_lower= 1/ K_upper
kappa_est = np.random.uniform(1, 100)
ts = np.linspace(0, T, int(T/dt)+1)
simga = 0.01

lambda_buy, lambda_sell = 0.1, 0.1
phi = 1e-4
Regret_MC = 200
start = 10
 
const = np.zeros((len(q_upper_list), len(q_lower_list)))
const_2 = np.zeros((len(q_upper_list), len(q_lower_list)))

In [ ]:
for idxqbar, q_upper in enumerate(q_upper_list):
    for idxqunder, q_lower in enumerate(q_lower_list):
        regret_list = []

        for _ in range(Regret_MC):
            agent = Agent(
                lambda_buy=lambda_buy,
                lambda_sell=lambda_sell,
                q_upper=q_upper,
                q_lower=q_lower,
                phi=phi,
                kappa_est=kappa_est,
                T=T,
                dt=dt,
                K_upper=K_upper,
                K_lower=K_lower, 
            )

            agent.learning(sigma=simga, kappa_true=kappa_true)
            regret_list.append(agent.regret(kappa_true=kappa_true))

        regret_list = np.array(regret_list)
        y = np.mean(regret_list, axis=0)
        y_axis = (y)[start:]
        x_axis = ts[start:]

        # Use log^2 model fit curve
        params, covariance = curve_fit(log_square_model, x_axis, y_axis)
        const[idxqbar, idxqunder] = params[0]
        const_2[idxqbar, idxqunder] = params[1]

In [ ]:
df = pd.DataFrame(const, index=q_upper_list, columns=q_lower_list)
df.to_csv("C1_qbar_qlower.csv", index=False)

In [ ]:
# Enable LaTeX rendering
plt.rc('text', usetex=True)

# Plot the heatmap
sns.heatmap(df, cmap='YlGnBu', annot=False, vmin=np.min(const), vmax=np.max(const))

# Set axis labels
plt.xlabel(r"$\underline{q}$")  # Use raw string (r"") for LaTeX commands
plt.ylabel(r"$\bar{q}$")        # Bar q in the y-axis label
plt.title(r"Dependency of fitted $C_1$ on model parameters")
# plt.show()

plt.savefig("/Users/galen/Desktop/C1_qupper_qlower.pdf", format='pdf', bbox_inches='tight')
plt.close()